Note: We are doing a BNP-esque method for dynamic topics using HDBSCAN since it is a bit hard to implement DDP

# Setup

In [1]:
from pathlib import Path
import sqlite3
import pickle

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = PROJECT_ROOT / "db" / "app.db"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print("DB_PATH exists:", DB_PATH.exists(), DB_PATH)
print("EMBEDDING_MODEL_NAME:", EMBEDDING_MODEL_NAME)

DB_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/app.db
EMBEDDING_MODEL_NAME: all-MiniLM-L6-v2


In [2]:
# Load Documents
conn = sqlite3.connect(DB_PATH)

docs_df = pd.read_sql_query("""
    SELECT
        d.id AS document_id,
        d.publication_year,
        d.title,
        d.clean_text,
        e.embedding
    FROM documents d
    JOIN document_embeddings e
        ON d.id = e.document_id
    WHERE e.model_name = ?
    ORDER BY d.publication_year, d.id
""", conn, params=[EMBEDDING_MODEL_NAME])

conn.close()

docs_df["embedding"] = docs_df["embedding"].apply(
    lambda x: np.array(pickle.loads(x), dtype=np.float32)
)

print("Loaded rows:", len(docs_df))
display(docs_df.head())

Loaded rows: 738


,document_id,publication_year,title,clean_text,embedding
0,1,2020,Study protocol: Comparison of different risk p...,Study protocol: Comparison of different risk p...,"[-0.022171408, -0.045305148, -0.035417516, 0.0..."
1,2,2020,"Mental health, sleep quality and quality of li...","Mental health, sleep quality and quality of li...","[0.054407, 0.02136461, -0.021674903, 0.0654725..."
2,3,2020,Parent perspectives on food allergy management...,Parent perspectives on food allergy management...,"[0.03656542, 0.027708251, 0.04394856, 0.021187..."
3,4,2020,COVID-19 critical illness in Sweden: character...,COVID-19 critical illness in Sweden: character...,"[0.060955044, -0.018265247, -0.056825712, -0.0..."
4,5,2020,Geolocated Twitter-based population mobility i...,Geolocated Twitter-based population mobility i...,"[0.097378224, -0.005816912, 0.026064621, 0.034..."


In [3]:
# See available years
year_counts = docs_df["publication_year"].value_counts().sort_index()

print("Documents per year:")
display(year_counts)

Documents per year:


publication_year
2020    450
2021    261
2022     27
Name: count, dtype: int64

# Conservative HDBSCAN test

In [4]:
# Normalize
from sklearn.preprocessing import normalize

docs_df["embedding_norm"] = list(
    normalize(np.vstack(docs_df["embedding"].values), norm="l2")
)

print("Added normalized embeddings.")
display(docs_df[["document_id", "publication_year", "embedding_norm"]].head())

Added normalized embeddings.


,document_id,publication_year,embedding_norm
0,1,2020,"[-0.022171406, -0.045305144, -0.035417512, 0.0..."
1,2,2020,"[0.054407, 0.02136461, -0.021674903, 0.0654725..."
2,3,2020,"[0.03656542, 0.027708251, 0.04394856, 0.021187..."
3,4,2020,"[0.060955044, -0.018265247, -0.056825712, -0.0..."
4,5,2020,"[0.09737823, -0.0058169123, 0.026064623, 0.034..."


In [8]:
# Import DBSCAN and set params
import hdbscan

HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 2

print("HDBSCAN_MIN_CLUSTER_SIZE:", HDBSCAN_MIN_CLUSTER_SIZE)
print("HDBSCAN_MIN_SAMPLES:", HDBSCAN_MIN_SAMPLES)

HDBSCAN_MIN_CLUSTER_SIZE: 5
HDBSCAN_MIN_SAMPLES: 2


In [9]:
# Cluster each year independently
yearly_clustered = []

for year in sorted(docs_df["publication_year"].unique()):
    year_df = docs_df[docs_df["publication_year"] == year].copy()
    X_year = np.vstack(year_df["embedding_norm"].values)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean"
    )

    labels = clusterer.fit_predict(X_year)

    year_df["hdbscan_label"] = labels
    yearly_clustered.append(year_df)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())

    print(
        f"Year {year} | docs={len(year_df)} | "
        f"clusters={n_clusters} | noise_docs={n_noise}"
    )

hdbscan_df = pd.concat(yearly_clustered, ignore_index=True)

print("\nCombined shape:", hdbscan_df.shape)
display(hdbscan_df.head())

Year 2020 | docs=450 | clusters=5 | noise_docs=309
Year 2021 | docs=261 | clusters=4 | noise_docs=109
Year 2022 | docs=27 | clusters=0 | noise_docs=27

Combined shape: (738, 7)


,document_id,publication_year,title,clean_text,embedding,embedding_norm,hdbscan_label
0,1,2020,Study protocol: Comparison of different risk p...,Study protocol: Comparison of different risk p...,"[-0.022171408, -0.045305148, -0.035417516, 0.0...","[-0.022171406, -0.045305144, -0.035417512, 0.0...",3
1,2,2020,"Mental health, sleep quality and quality of li...","Mental health, sleep quality and quality of li...","[0.054407, 0.02136461, -0.021674903, 0.0654725...","[0.054407, 0.02136461, -0.021674903, 0.0654725...",2
2,3,2020,Parent perspectives on food allergy management...,Parent perspectives on food allergy management...,"[0.03656542, 0.027708251, 0.04394856, 0.021187...","[0.03656542, 0.027708251, 0.04394856, 0.021187...",-1
3,4,2020,COVID-19 critical illness in Sweden: character...,COVID-19 critical illness in Sweden: character...,"[0.060955044, -0.018265247, -0.056825712, -0.0...","[0.060955044, -0.018265247, -0.056825712, -0.0...",-1
4,5,2020,Geolocated Twitter-based population mobility i...,Geolocated Twitter-based population mobility i...,"[0.097378224, -0.005816912, 0.026064621, 0.034...","[0.09737823, -0.0058169123, 0.026064623, 0.034...",1


In [10]:
# Inspect Label Counts by year
for year in sorted(hdbscan_df["publication_year"].unique()):
    print(f"\n=== Year {year} ===")
    counts = (
        hdbscan_df[hdbscan_df["publication_year"] == year]["hdbscan_label"]
        .value_counts()
        .sort_index()
    )
    display(counts)


=== Year 2020 ===


hdbscan_label
-1    309
 0      6
 1     11
 2     31
 3     84
 4      9
Name: count, dtype: int64


=== Year 2021 ===


hdbscan_label
-1    109
 0      7
 1      8
 2    124
 3     13
Name: count, dtype: int64


=== Year 2022 ===


hdbscan_label
-1    27
Name: count, dtype: int64

We tried less conservative clusters and 2022 still gives one. We will assume these as emerging topics rather than forcing 2022 to cluster

# Yearly Topic Representations

In [11]:
# Filter out noise
clustered_df = hdbscan_df[hdbscan_df["hdbscan_label"] != -1].copy()

print("Clustered docs:", len(clustered_df))
display(clustered_df.head())

Clustered docs: 293


,document_id,publication_year,title,clean_text,embedding,embedding_norm,hdbscan_label
0,1,2020,Study protocol: Comparison of different risk p...,Study protocol: Comparison of different risk p...,"[-0.022171408, -0.045305148, -0.035417516, 0.0...","[-0.022171406, -0.045305144, -0.035417512, 0.0...",3
1,2,2020,"Mental health, sleep quality and quality of li...","Mental health, sleep quality and quality of li...","[0.054407, 0.02136461, -0.021674903, 0.0654725...","[0.054407, 0.02136461, -0.021674903, 0.0654725...",2
4,5,2020,Geolocated Twitter-based population mobility i...,Geolocated Twitter-based population mobility i...,"[0.097378224, -0.005816912, 0.026064621, 0.034...","[0.09737823, -0.0058169123, 0.026064623, 0.034...",1
8,9,2020,[COVID-19 and its socio-cultural imagery in La...,[COVID-19 and its socio-cultural imagery in La...,"[0.11588997, 0.060065616, -0.08925954, 0.02039...","[0.115889974, 0.06006562, -0.08925955, 0.02039...",3
9,10,2020,"[Risks, contamination and prevention against C...","[Risks, contamination and prevention against C...","[-0.01211118, -0.016977603, -0.048181903, -0.0...","[-0.012111182, -0.016977604, -0.04818191, -0.0...",2


In [12]:
# compute centroids
centroids = []

for (year, label), group in clustered_df.groupby(["publication_year", "hdbscan_label"]):
    X = np.vstack(group["embedding_norm"].values)

    centroid = X.mean(axis=0)
    centroid = centroid / np.linalg.norm(centroid)

    centroids.append({
        "publication_year": year,
        "cluster_id": int(label),
        "centroid": centroid.astype(np.float32),
        "n_docs": len(group),
    })

centroids_df = pd.DataFrame(centroids)

print("Centroids shape:", centroids_df.shape)
display(centroids_df.head())

Centroids shape: (9, 4)


,publication_year,cluster_id,centroid,n_docs
0,2020,0,"[-0.0062895324, -0.0037084273, 0.0053543854, -...",6
1,2020,1,"[0.019295102, -0.0031670234, 0.031987768, -0.0...",11
2,2020,2,"[0.0062654614, 0.050295692, -0.043644834, 0.04...",31
3,2020,3,"[0.0065431446, 0.010224107, -0.029027194, -0.0...",84
4,2020,4,"[-0.021744326, 0.004247416, 0.004051477, 0.008...",9


In [13]:
# cluster sizes
display(
    centroids_df.sort_values(["publication_year", "n_docs"], ascending=[True, False])
)

,publication_year,cluster_id,centroid,n_docs
3,2020,3,"[0.0065431446, 0.010224107, -0.029027194, -0.0...",84
2,2020,2,"[0.0062654614, 0.050295692, -0.043644834, 0.04...",31
1,2020,1,"[0.019295102, -0.0031670234, 0.031987768, -0.0...",11
4,2020,4,"[-0.021744326, 0.004247416, 0.004051477, 0.008...",9
0,2020,0,"[-0.0062895324, -0.0037084273, 0.0053543854, -...",6
7,2021,2,"[-0.00059794314, 0.020194774, -0.025267798, -0...",124
8,2021,3,"[-0.010735666, -0.008001255, 0.016922465, -0.0...",13
6,2021,1,"[0.03604189, -0.054909248, 0.00085565116, 0.05...",8
5,2021,0,"[-0.019979605, 0.026301166, 0.03203754, -0.080...",7


# Persistence, Birth, and Death of Topics from 2020-2021

In [14]:
# Similarity matrix
from sklearn.metrics.pairwise import cosine_similarity

centroids_2020 = centroids_df[centroids_df["publication_year"] == 2020].copy()
centroids_2021 = centroids_df[centroids_df["publication_year"] == 2021].copy()

X_2020 = np.vstack(centroids_2020["centroid"].values)
X_2021 = np.vstack(centroids_2021["centroid"].values)

sim_matrix = cosine_similarity(X_2020, X_2021)

print("Similarity matrix shape:", sim_matrix.shape)

sim_df = pd.DataFrame(
    sim_matrix,
    index=[f"2020_{cid}" for cid in centroids_2020["cluster_id"]],
    columns=[f"2021_{cid}" for cid in centroids_2021["cluster_id"]],
)

display(sim_df)

Similarity matrix shape: (5, 4)


,2021_0,2021_1,2021_2,2021_3
2020_0,0.240711,0.286877,0.629670,0.913978
2020_1,0.208353,0.419652,0.547259,0.347370
2020_2,0.251753,0.499039,0.828841,0.358450
2020_3,0.327068,0.512035,0.954569,0.536466
2020_4,0.326214,0.467717,0.632914,0.380070


In [15]:
# Greedy matchiing
matches = []

for i, row in enumerate(sim_matrix):
    best_j = np.argmax(row)
    best_sim = row[best_j]

    matches.append({
        "cluster_2020": int(centroids_2020.iloc[i]["cluster_id"]),
        "cluster_2021": int(centroids_2021.iloc[best_j]["cluster_id"]),
        "similarity": float(best_sim),
    })

matches_df = pd.DataFrame(matches)

display(matches_df.sort_values("similarity", ascending=False))

,cluster_2020,cluster_2021,similarity
3,3,2,0.954569
0,0,3,0.913978
2,2,2,0.828841
4,4,2,0.632914
1,1,2,0.547259


In [16]:
# persistence vs death
SIM_THRESHOLD = 0.8

matches_df["status"] = matches_df["similarity"].apply(
    lambda x: "persistent" if x >= SIM_THRESHOLD else "weak_match"
)

display(matches_df.sort_values("similarity"))

,cluster_2020,cluster_2021,similarity,status
1,1,2,0.547259,weak_match
4,4,2,0.632914,weak_match
2,2,2,0.828841,persistent
0,0,3,0.913978,persistent
3,3,2,0.954569,persistent


In [17]:
# detect births
matched_2021 = set(matches_df["cluster_2021"])

all_2021 = set(centroids_2021["cluster_id"])

unmatched_2021 = all_2021 - matched_2021

births_df = centroids_2021[
    centroids_2021["cluster_id"].isin(unmatched_2021)
].copy()

births_df["status"] = "birth"

print("New topics in 2021 (births):")
display(births_df)

New topics in 2021 (births):


,publication_year,cluster_id,centroid,n_docs,status
5,2021,0,"[-0.019979605, 0.026301166, 0.03203754, -0.080...",7,birth
6,2021,1,"[0.03604189, -0.054909248, 0.00085565116, 0.05...",8,birth


We get the following results like what DPs would produce, but from geometry instead: Many to one mappings represent topic merging, and new clusters/disappearing clusters are birth and death of topics

# Linneage Creation

In [23]:
# Initialize 2020 linneage
lineage_records = []

next_lineage_id = 0

lineage_map_2020 = {}

for _, row in centroids_2020.iterrows():
    cid = int(row["cluster_id"])

    lineage_map_2020[cid] = next_lineage_id

    lineage_records.append({
        "lineage_id": next_lineage_id,
        "year": 2020,
        "cluster_id": cid,
        "n_docs": row["n_docs"],
    })

    next_lineage_id += 1

print("Initial lineage count:", next_lineage_id)

Initial lineage count: 5


In [24]:
# Propogate to 2021
lineage_map_2021 = {}

for _, row in matches_df.iterrows():
    cid_2020 = int(row["cluster_2020"])
    cid_2021 = int(row["cluster_2021"])
    sim = row["similarity"]

    if sim >= SIM_THRESHOLD:
        lineage_id = lineage_map_2020[cid_2020]
        lineage_map_2021[cid_2021] = lineage_id
    else:
        # weak match: do not propagate
        continue

In [25]:
# Assign births to new linneage
for _, row in births_df.iterrows():
    cid = int(row["cluster_id"])

    lineage_map_2021[cid] = next_lineage_id

    lineage_records.append({
        "lineage_id": next_lineage_id,
        "year": 2021,
        "cluster_id": cid,
        "n_docs": row["n_docs"],
    })

    next_lineage_id += 1

In [26]:
records_2021 = []

for _, row in centroids_2021.iterrows():
    cid = int(row["cluster_id"])

    if cid in lineage_map_2021:
        lineage_id = lineage_map_2021[cid]

        records_2021.append({
            "lineage_id": lineage_id,
            "year": 2021,
            "cluster_id": cid,
            "n_docs": row["n_docs"],
        })

# Remove any previously added 2021 rows (from births step)
lineage_records = [
    r for r in lineage_records if r["year"] != 2021
]

# Add clean 2021 records
lineage_records.extend(records_2021)

lineage_df = pd.DataFrame(lineage_records)

display(lineage_df.sort_values(["lineage_id", "year"]))

,lineage_id,year,cluster_id,n_docs
0,0,2020,0,6
8,0,2021,3,13
1,1,2020,1,11
2,2,2020,2,31
3,3,2020,3,84
7,3,2021,2,124
4,4,2020,4,9
5,5,2021,0,7
6,6,2021,1,8


In [27]:
# Save linneage
lineage_path = PROJECT_ROOT / "data" / "hdbscan_lineage.csv"
lineage_df.to_csv(lineage_path, index=False)

print("Saved lineage to:", lineage_path)

Saved lineage to: /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/data/hdbscan_lineage.csv


We now have two approaches built: Fixed topics with KMeans+Diffusion and Variable Topics with HDBSCAN